# Train TextCNN v3 (`data_ready_v1_3`)

Cùng cấu trúc với `pipeline_v1.1/train-textcnn_v2.ipynb`, nhưng:
- Data: **`data/data_ready_v1_3`** (6 `macro_domain`; 3 lớp hiếm train đã gộp vào `other`).
- Chuẩn bị data: `python pipeline_v1.3/prepare_data.py --out-dir data/data_ready_v1_3`
- Embedding: `python pipeline_v1.3/build_shared_embedding.py` → `pipeline_v1.3/shared_embedding_artifacts/`.
- Class weights: gamma cao hơn cho **`other`** (thay cho *Civil & Investment* ở v2).
- Artifacts: `artifacts/textcnn_v1_3` (local) hoặc `/kaggle/working/textcnn_artifacts_v1_3`.


In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset


def load_artifacts(artifact_dir):
    artifact_dir = Path(artifact_dir)
    with open(artifact_dir / "tokenizer_vocab.json", "r", encoding="utf-8") as f:
        vocab = json.load(f)
    ckpt = torch.load(artifact_dir / "embedding.pt", map_location="cpu")
    stoi = vocab["stoi"]
    embedding_weight = ckpt["embedding_weight"]
    pad_idx = int(ckpt["pad_idx"])
    return stoi, embedding_weight, pad_idx


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True


In [ ]:
def detect_data_dir():
    candidates = [
        Path("data/data_ready_v1_3"),
        Path("../data/data_ready_v1_3"),
        Path("/kaggle/input/vnlegal-rag/data/data_ready_v1_3"),
        Path("/kaggle/working/vnlegal-rag/data/data_ready_v1_3"),
        Path("/kaggle/input/datasets/hngphtrn/legals/data_ready_v1_3"),
        Path("/kaggle/input/datasets/hngphtrn/legals-v1-3"),
        Path("/kaggle/input/datasets/hngphtrn/legals_v1_3"),
    ]
    for p in candidates:
        if p.exists() and (p / "qa_train.csv").exists():
            return p.resolve()
    raise FileNotFoundError(
        "Cannot find data_ready_v1_3 with qa_train.csv. "
        "Run: python pipeline_v1.3/prepare_data.py --out-dir data/data_ready_v1_3"
    )


DATA_DIR = detect_data_dir()
ARTIFACT_DIR = Path(
    "/kaggle/working/textcnn_artifacts_v1_3"
    if Path("/kaggle").exists()
    else "artifacts/textcnn_v1_3"
)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR =", DATA_DIR)
print("ARTIFACT_DIR =", ARTIFACT_DIR)


In [ ]:
qa_train = pd.read_csv(DATA_DIR / "qa_train.csv", sep="\t")
qa_val = pd.read_csv(DATA_DIR / "qa_val.csv", sep="\t")
qa_test = pd.read_csv(DATA_DIR / "qa_test.csv", sep="\t")

required_cols = {"question", "macro_domain"}
for name, df in [("train", qa_train), ("val", qa_val), ("test", qa_test)]:
    miss = required_cols - set(df.columns)
    if miss:
        raise ValueError(f"{name} missing columns: {miss}")

print("train:", qa_train.shape, "val:", qa_val.shape, "test:", qa_test.shape)
qa_train[["question", "macro_domain"]].head()


In [ ]:
# pyvi / underthesea không cần — pipeline dùng simple_tokenize (regex \w+).


In [ ]:
import re


def simple_tokenize(text: str) -> list:
    r"""Tokenizer chuẩn: regex \w+, lowercase, Unicode-aware."""
    return re.findall(r"\w+", str(text or "").lower().strip(), flags=re.UNICODE)


def tokenize(text: str) -> list:
    return simple_tokenize(text)


TOKENIZER_BACKEND = "simple_tokenize"
print(f"Tokenizer backend: {TOKENIZER_BACKEND}")
print("  regex \\w+ — no pyvi / underthesea required")


In [ ]:
PAD = "<PAD>"
UNK = "<UNK>"
MAX_LEN = 128

_embed_candidates = [
    Path("/kaggle/input/datasets/hngphtrn/legal-embedding-v1-3"),
    Path("pipeline_v1.3/shared_embedding_artifacts"),
    Path("../pipeline_v1.3/shared_embedding_artifacts"),
]
SHARED_EMBED_DIR = None
for _p in _embed_candidates:
    if _p.exists() and (_p / "embedding.pt").exists():
        SHARED_EMBED_DIR = _p
        break
if SHARED_EMBED_DIR is None:
    raise FileNotFoundError(
        "No shared embedding dir with embedding.pt. Run: python pipeline_v1.3/build_shared_embedding.py"
    )

print("TOKENIZER_BACKEND=", TOKENIZER_BACKEND)
print("MAX_LEN=", MAX_LEN)
print("SHARED_EMBED_DIR=", SHARED_EMBED_DIR)

stoi, embedding_weight, _pad_ckpt = load_artifacts(SHARED_EMBED_DIR)
itos = {i: w for w, i in stoi.items()}
MAX_VOCAB = len(stoi)

token_lens = [len(simple_tokenize(q)) for q in qa_train["question"].astype(str).tolist()]
max_len_tokens = max(token_lens)
p95_len = float(np.percentile(token_lens, 95))
p99_len = float(np.percentile(token_lens, 99))
pct_truncated = 100.0 * sum(1 for L in token_lens if L > MAX_LEN) / len(token_lens)

print(
    "max_len_tokens:", max_len_tokens,
    "| p95:", round(p95_len, 2),
    "| p99:", round(p99_len, 2),
)
print(f"pct samples truncated (len > MAX_LEN={MAX_LEN}): {pct_truncated:.2f}%")
print("Loaded shared vocab size:", len(stoi), "| embedding shape:", tuple(embedding_weight.shape))

_label_maps_path = DATA_DIR / "label_maps.json"
if _label_maps_path.exists():
    with open(_label_maps_path, "r", encoding="utf-8") as _f:
        _lm = json.load(_f)
    labels = list(_lm["label_list"])
    label2id = {str(k): int(v) for k, v in _lm["label2id"].items()}
    label2id = {k: label2id[k] for k in labels}
    id2label = {int(k): v for k, v in _lm["id2label"].items()}
else:
    labels = sorted(qa_train["macro_domain"].unique().tolist())
    label2id = {l: i for i, l in enumerate(labels)}
    id2label = {i: l for l, i in label2id.items()}

print("Vocab size:", len(stoi), "| Num labels:", len(labels))


In [ ]:
def encode_text(text, max_len=MAX_LEN):
    tokens = simple_tokenize(text)
    ids = [stoi.get(t, stoi["<UNK>"]) for t in tokens][:max_len]
    if len(ids) < max_len:
        ids += [stoi["<PAD>"]] * (max_len - len(ids))
    return ids


class QADomainDataset(Dataset):
    def __init__(self, df):
        self.questions = df["question"].astype(str).tolist()
        self.labels = [label2id[x] for x in df["macro_domain"].tolist()]

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        return (
            torch.tensor(encode_text(self.questions[idx]), dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long),
        )


In [ ]:
BATCH_SIZE = 50
import sys

_num_workers = 0 if sys.platform == "win32" else 2
_pin = torch.cuda.is_available()

train_ds = QADomainDataset(qa_train)
val_ds = QADomainDataset(qa_val)
test_ds = QADomainDataset(qa_test)

train_label_ids = np.array([label2id[x] for x in qa_train["macro_domain"].tolist()])
class_counts = np.bincount(train_label_ids, minlength=len(labels))

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=_num_workers, pin_memory=_pin
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=_num_workers, pin_memory=_pin
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=_num_workers, pin_memory=_pin
)

print(len(train_ds), len(val_ds), len(test_ds))
print("Class counts:", class_counts.tolist())


In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes, filter_sizes=(3, 4, 5), num_filters=100, dropout=0.5, embed_dropout=0.0, pad_idx=0, embedding_weight=None):
        super().__init__()
        if embedding_weight is None:
            self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        else:
            self.embedding = nn.Embedding.from_pretrained(embedding_weight, freeze=False, padding_idx=pad_idx)
        self.embed_dropout = nn.Dropout(embed_dropout) if embed_dropout and embed_dropout > 0 else None
        self.convs = nn.ModuleList([nn.Conv1d(embed_dim, num_filters, k) for k in filter_sizes])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        if self.embed_dropout is not None:
            emb = self.embed_dropout(emb)
        emb = emb.transpose(1, 2)
        feats = [F.relu(conv(emb)) for conv in self.convs]
        pooled = [F.max_pool1d(f, kernel_size=f.shape[2]).squeeze(2) for f in feats]
        out = torch.cat(pooled, dim=1)
        out = self.dropout(out)
        return self.fc(out)


class FocalLoss(nn.Module):
    """Multi-class focal loss + optional class weights (same as v2)."""

    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.gamma = float(gamma)
        self.label_smoothing = float(label_smoothing)
        if weight is not None:
            self.register_buffer("class_weight", weight)
        else:
            self.class_weight = None

    def forward(self, logits, targets):
        w = self.class_weight
        ce = F.cross_entropy(logits, targets, weight=w, reduction="none", label_smoothing=self.label_smoothing)
        pt = F.softmax(logits.float(), dim=1).gather(1, targets.unsqueeze(1)).squeeze(1).clamp_min(1e-8)
        mod = (1.0 - pt) ** self.gamma
        return (mod * ce).mean()


In [ ]:
MAX_NORM_FC = 3.0
WEIGHT_DECAY = 1e-4
TEXTCNN_DROPOUT = 0.55
EMBED_DROPOUT = 0.2
LABEL_SMOOTHING = 0.05
USE_FOCAL = False  # True: FocalLoss (thử nếu cần)
FOCAL_GAMMA = 2.0
BASE_LR = 1.0
EMBED_LR_SCALE = 0.35
EMBED_FREEZE_EPOCHS = 0

MINORITY_EXTRA_LABEL = "other"  # v1.3: bucket gộp 3 lớp hiếm
CE_MINORITY_GAMMA_DEFAULT = 1.5
CE_MINORITY_GAMMA_EXTRA = 2.0

model = TextCNN(
    vocab_size=len(stoi),
    embed_dim=int(embedding_weight.shape[1]),
    num_classes=len(labels),
    filter_sizes=(2, 3, 4, 5),
    num_filters=128,
    dropout=TEXTCNN_DROPOUT,
    embed_dropout=EMBED_DROPOUT,
    pad_idx=stoi["<PAD>"],
    embedding_weight=embedding_weight,
).to(device)

_counts = torch.as_tensor(class_counts, dtype=torch.float32, device=device).clamp(min=1.0)
_inv_balanced = _counts.sum() / (len(labels) * _counts)
_gammas = torch.full((len(labels),), CE_MINORITY_GAMMA_DEFAULT, dtype=torch.float32, device=device)
if MINORITY_EXTRA_LABEL in label2id:
    _gammas[label2id[MINORITY_EXTRA_LABEL]] = CE_MINORITY_GAMMA_EXTRA
else:
    print("[warn] MINORITY_EXTRA_LABEL not in label2id; uniform gammas.")
class_weights = _inv_balanced.pow(_gammas)
class_weights = class_weights * (len(labels) / class_weights.sum())
if USE_FOCAL:
    criterion = FocalLoss(class_weights, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)
else:
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
criterion = criterion.to(device)
print(
    ("FocalLoss" if USE_FOCAL else "CrossEntropyLoss"),
    f"gamma={FOCAL_GAMMA}" if USE_FOCAL else "",
    "| class_weights:",
    {lab: round(w.item(), 4) for lab, w in zip(labels, class_weights)},
)

embedding_params = list(model.embedding.parameters())
embedding_param_ids = {id(p) for p in embedding_params}
base_params = [p for p in model.parameters() if id(p) not in embedding_param_ids]

optimizer = torch.optim.Adadelta(
    [
        {"params": base_params, "lr": BASE_LR, "rho": 0.95, "eps": 1e-6, "weight_decay": WEIGHT_DECAY},
        {"params": embedding_params, "lr": BASE_LR * EMBED_LR_SCALE, "rho": 0.95, "eps": 1e-6, "weight_decay": WEIGHT_DECAY},
    ]
)

scaler = GradScaler("cuda", enabled=torch.cuda.is_available())
print(f"Embedding schedule: freeze {EMBED_FREEZE_EPOCHS} epochs, embed_lr={BASE_LR * EMBED_LR_SCALE:.3f}")


In [ ]:
def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if train:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(train):
            with autocast(device_type="cuda", enabled=torch.cuda.is_available()):
                logits = model(x)
                loss = criterion(logits, y)
            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                with torch.no_grad():
                    w = model.fc.weight
                    norm = w.norm(2)
                    if norm > MAX_NORM_FC:
                        w.mul_(MAX_NORM_FC / (norm + 1e-6))
        total_loss += loss.item() * x.size(0)
        pred = torch.argmax(logits, dim=1)
        y_true.extend(y.detach().cpu().tolist())
        y_pred.extend(pred.detach().cpu().tolist())
    avg_loss = total_loss / len(loader.dataset)
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    return avg_loss, macro_f1, y_true, y_pred


In [ ]:
EPOCHS = 20
PATIENCE = 5
EMBED_FREEZE_EPOCHS = int(globals().get("EMBED_FREEZE_EPOCHS", 0))
embedding_params = list(globals().get("embedding_params", model.embedding.parameters()))
EARLY_STOP_METRIC = "composite"
VAL_LOSS_WEIGHT = 0.12
MIN_DELTA = 1e-5


def _val_monitor(va_f1: float, va_loss: float) -> float:
    if EARLY_STOP_METRIC == "val_f1":
        return float(va_f1)
    if EARLY_STOP_METRIC == "val_loss":
        return -float(va_loss)
    return float(va_f1) - VAL_LOSS_WEIGHT * float(va_loss)


best_monitor = float("-inf")
best_val_f1_at_ckpt = 0.0
best_val_loss_at_ckpt = float("inf")
wait = 0
best_path = ARTIFACT_DIR / "textcnn_best.pt"
history = []

for epoch in range(1, EPOCHS + 1):
    freeze_embedding = epoch <= EMBED_FREEZE_EPOCHS
    for p in embedding_params:
        p.requires_grad = not freeze_embedding
    tr_loss, tr_f1, _, _ = run_epoch(model, train_loader, optimizer=optimizer)
    va_loss, va_f1, _, _ = run_epoch(model, val_loader, optimizer=None)
    monitor = _val_monitor(va_f1, va_loss)
    history.append({
        "epoch": epoch,
        "train_loss": tr_loss,
        "train_f1": tr_f1,
        "val_loss": va_loss,
        "val_f1": va_f1,
        "val_monitor": monitor,
        "freeze_embedding": freeze_embedding,
    })
    phase = "frozen" if freeze_embedding else "finetune"
    print(
        f"Epoch {epoch:02d} [{phase}] | train_loss={tr_loss:.4f} train_f1={tr_f1:.4f} | "
        f"val_loss={va_loss:.4f} val_f1={va_f1:.4f} | monitor={monitor:.4f}"
    )
    if monitor > best_monitor + MIN_DELTA:
        best_monitor = monitor
        best_val_f1_at_ckpt = va_f1
        best_val_loss_at_ckpt = va_loss
        wait = 0
        torch.save(model.state_dict(), best_path)
    else:
        wait += 1
        if wait >= PATIENCE:
            print("Early stopping triggered.")
            break

print(
    f"Best checkpoint | metric={EARLY_STOP_METRIC} monitor={round(best_monitor, 4)} | "
    f"val_f1={round(best_val_f1_at_ckpt, 4)} val_loss={round(best_val_loss_at_ckpt, 4)}"
)


In [ ]:
model.load_state_dict(torch.load(best_path, map_location=device))
te_loss, te_f1, y_true, y_pred = run_epoch(model, test_loader, optimizer=None)
print("Test loss:", round(te_loss, 4), "| Test macro F1:", round(te_f1, 4))
print(classification_report(y_true, y_pred, target_names=labels, digits=4, zero_division=0))


In [ ]:
metadata_out = {
    "version": "textcnn_v1_3",
    "tokenizer_backend": TOKENIZER_BACKEND,
    "shared_embed_dir": str(SHARED_EMBED_DIR),
    "data_dir": str(DATA_DIR),
    "max_len": MAX_LEN,
    "max_vocab_cap": MAX_VOCAB,
    "labels": labels,
    "label2id": label2id,
    "id2label": {str(k): v for k, v in id2label.items()},
    "pad_token": PAD,
    "unk_token": UNK,
    "train_strategy": {
        "sampling": "shuffle",
        "loss": "focal_class_weighted" if USE_FOCAL else "cross_entropy_class_weighted",
        "focal_gamma": FOCAL_GAMMA if USE_FOCAL else None,
        "minority_extra_label": MINORITY_EXTRA_LABEL,
        "ce_minority_gamma_default": CE_MINORITY_GAMMA_DEFAULT,
        "ce_minority_gamma_extra": CE_MINORITY_GAMMA_EXTRA,
        "ce_class_weights": {lab: round(float(w), 6) for lab, w in zip(labels, class_weights.detach().cpu())},
        "optimizer": "adadelta",
        "adadelta_lr_base": BASE_LR,
        "adadelta_lr_embed": BASE_LR * EMBED_LR_SCALE,
        "adadelta_rho": 0.95,
        "adadelta_eps": 1e-6,
        "embed_freeze_epochs": EMBED_FREEZE_EPOCHS,
        "batch_size": BATCH_SIZE,
        "grad_clip": "none",
        "max_norm_fc": MAX_NORM_FC,
        "label_smoothing": LABEL_SMOOTHING,
        "weight_decay": WEIGHT_DECAY,
        "dropout": TEXTCNN_DROPOUT,
        "embed_dropout": EMBED_DROPOUT,
        "scheduler": "none",
        "early_stop_metric": EARLY_STOP_METRIC,
        "early_stop_patience": PATIENCE,
        "early_stop_min_delta": MIN_DELTA,
        "early_stop_val_loss_weight": VAL_LOSS_WEIGHT if EARLY_STOP_METRIC == "composite" else None,
        "best_val_monitor": round(float(best_monitor), 6),
        "best_val_f1_at_ckpt": round(float(best_val_f1_at_ckpt), 6),
        "best_val_loss_at_ckpt": round(float(best_val_loss_at_ckpt), 6),
    },
}
with open(ARTIFACT_DIR / "tokenizer_vocab.json", "w", encoding="utf-8") as f:
    json.dump({"stoi": stoi, "itos": {str(k): v for k, v in itos.items()}}, f, ensure_ascii=False)
with open(ARTIFACT_DIR / "textcnn_meta.json", "w", encoding="utf-8") as f:
    json.dump(metadata_out, f, ensure_ascii=False, indent=2)
pd.DataFrame(history).to_csv(ARTIFACT_DIR / "train_history.csv", index=False)
print("Saved artifacts at:", ARTIFACT_DIR)
